# WP-Bench Results Report

This notebook loads benchmark results and generates visualizations.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

In [ ]:
# Load the most recent results file
output_dir = Path('../output')
result_files = sorted(output_dir.glob('results_*.json'), reverse=True)

if not result_files:
    raise FileNotFoundError('No results files found in output directory')

results_path = result_files[0]  # Most recent
print(f'Loading: {results_path.name}')

with open(results_path) as f:
    data = json.load(f)

In [ ]:
# Parse results into DataFrame
rows = []
for model_name, model_data in data['models'].items():
    scores = model_data['scores']
    rows.append({
        'Model': model_name,
        'Knowledge': scores['knowledge'] * 100,
        'Correctness': scores['correctness'] * 100,
        'Overall': scores['overall'] * 100,
    })

df = pd.DataFrame(rows)
df = df.sort_values('Overall', ascending=False).reset_index(drop=True)
df

In [ ]:
# Overall scores bar chart
fig = px.bar(
    df.sort_values('Overall'),
    x='Overall',
    y='Model',
    orientation='h',
    title='WP-Bench Overall Scores',
    labels={'Overall': 'Score (%)', 'Model': ''},
    color='Overall',
    color_continuous_scale='Blues'
)
fig.update_layout(showlegend=False, height=400)
fig.show()

In [ ]:
# Side-by-side comparison of Knowledge vs Correctness
df_melted = df.melt(
    id_vars=['Model'],
    value_vars=['Knowledge', 'Correctness'],
    var_name='Metric',
    value_name='Score'
)

fig = px.bar(
    df_melted,
    x='Model',
    y='Score',
    color='Metric',
    barmode='group',
    title='Knowledge vs Correctness by Model',
    labels={'Score': 'Score (%)', 'Model': ''},
    color_discrete_map={'Knowledge': '#3498db', 'Correctness': '#e74c3c'}
)
fig.update_layout(height=450, xaxis_tickangle=-45)
fig.show()

In [ ]:
# Radar chart for top models
top_models = df.head(4)

fig = go.Figure()

categories = ['Knowledge', 'Correctness', 'Overall']

for _, row in top_models.iterrows():
    values = [row['Knowledge'], row['Correctness'], row['Overall']]
    values.append(values[0])  # Close the radar
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        name=row['Model'],
        fill='toself',
        opacity=0.6
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    showlegend=True,
    title='Top 4 Models Comparison',
    height=500
)
fig.show()

In [ ]:
# Styled summary table
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #90EE90' if v else '' for v in is_max]

styled = df.style.apply(highlight_max, subset=['Knowledge', 'Correctness', 'Overall'])
styled = styled.format({
    'Knowledge': '{:.1f}%',
    'Correctness': '{:.1f}%',
    'Overall': '{:.1f}%'
})
styled

In [ ]:
# Export to standalone HTML with embedded charts
from datetime import datetime

# Create overall scores chart
fig_overall = px.bar(
    df.sort_values('Overall'),
    x='Overall',
    y='Model',
    orientation='h',
    title='WP-Bench Overall Scores',
    labels={'Overall': 'Score (%)', 'Model': ''},
    color='Overall',
    color_continuous_scale='Blues'
)
fig_overall.update_layout(showlegend=False, height=400)

# Create comparison chart
df_melted = df.melt(
    id_vars=['Model'],
    value_vars=['Knowledge', 'Correctness'],
    var_name='Metric',
    value_name='Score'
)
fig_comparison = px.bar(
    df_melted,
    x='Model',
    y='Score',
    color='Metric',
    barmode='group',
    title='Knowledge vs Correctness by Model',
    labels={'Score': 'Score (%)', 'Model': ''},
    color_discrete_map={'Knowledge': '#3498db', 'Correctness': '#e74c3c'}
)
fig_comparison.update_layout(height=450, xaxis_tickangle=-45)

# Create radar chart
top_models = df.head(4)
fig_radar = go.Figure()
categories = ['Knowledge', 'Correctness', 'Overall']
for _, row in top_models.iterrows():
    values = [row['Knowledge'], row['Correctness'], row['Overall']]
    values.append(values[0])
    fig_radar.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        name=row['Model'],
        fill='toself',
        opacity=0.6
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    showlegend=True,
    title='Top 4 Models Comparison',
    height=500
)

# Convert charts to HTML divs
chart_overall = fig_overall.to_html(full_html=False, include_plotlyjs=False)
chart_comparison = fig_comparison.to_html(full_html=False, include_plotlyjs=False)
chart_radar = fig_radar.to_html(full_html=False, include_plotlyjs=False)

html_content = f'''
<!DOCTYPE html>
<html>
<head>
    <title>WP-Bench Results Report</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1200px; margin: 0 auto; padding: 20px; background: #fafafa; }}
        h1 {{ color: #333; border-bottom: 2px solid #4CAF50; padding-bottom: 10px; }}
        h2 {{ color: #555; margin-top: 30px; }}
        .metadata {{ color: #666; margin-bottom: 20px; background: #fff; padding: 15px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; background: #fff; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
        th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
        th {{ background-color: #4CAF50; color: white; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        tr:hover {{ background-color: #f1f1f1; }}
        .chart {{ margin: 30px 0; background: #fff; padding: 20px; border-radius: 8px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
    </style>
</head>
<body>
    <h1>WP-Bench Results Report</h1>
    <div class="metadata">
        <p><strong>Suite:</strong> {data['metadata']['suite']}</p>
        <p><strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        <p><strong>Source:</strong> {results_path.name}</p>
    </div>
    
    <h2>Overall Scores</h2>
    <div class="chart">{chart_overall}</div>
    
    <h2>Knowledge vs Correctness</h2>
    <div class="chart">{chart_comparison}</div>
    
    <h2>Top Models Comparison</h2>
    <div class="chart">{chart_radar}</div>
    
    <h2>Results Summary</h2>
    {df.to_html(index=False, float_format=lambda x: f'{x:.1f}%' if pd.notna(x) else 'N/A')}
</body>
</html>
'''

output_html = output_dir / f'report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.html'
output_html.write_text(html_content)
print(f'Report exported to: {output_html}')